# Multi-Agent Resume Matcher and Tailor Lab

This lab demonstrates a simple 4-agent pipeline using LM Studio and a Gradio interface.

**Agent 1 — Resume Reader:** extracts skills, experience, education, projects, and resume evidence.  
**Agent 2 — Job Posting Analyzer:** extracts required skills, preferred skills, responsibilities, and keywords.  
**Agent 3 — Match Evaluator:** compares the resume to the job posting and assigns a match score.  
**Agent 4 — Resume Tailor:** only rewrites the resume if the match score is at least the selected threshold, default **75%**.

Important classroom rule: the tailoring agent must **not invent experience**. It may reword, reorganize, and emphasize truthful evidence already found in the resume.


In [ ]:
# === Cell 1: Install Dependencies ===
# Run this cell first in a fresh kernel.

import sys
print("Python executable used by this notebook:", sys.executable)

%pip install --upgrade pip setuptools wheel
%pip install --upgrade \
    "gradio==5.49.1" \
    "gradio-client==1.13.3" \
    "pydantic>=2.11,<2.12" \
    "pandas>=2.2,<2.4" \
    "requests>=2.32,<3" \
    "pypdf>=5.0,<6" \
    "python-docx>=1.1,<2"


In [ ]:
# === Cell 2: Configure LM Studio and test connectivity ===

import os
import json
import requests

# Start LM Studio locally, load a chat/instruct model, and enable the local server.
os.environ["OPENAI_BASE_URL"] = "http://localhost:1234/v1"
os.environ["OPENAI_API_KEY"] = "lmstudio-key"  # LM Studio usually accepts any non-empty key.

# IMPORTANT:
# The model name must exactly match the model ID shown by LM Studio's /models endpoint.
# This cell auto-detects the first loaded model and uses it.
try:
    models_url = f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/models"
    resp = requests.get(models_url, timeout=10)
    print("LM Studio status:", resp.status_code)
    resp.raise_for_status()

    data = resp.json()
    print("Available model info:")
    print(json.dumps(data, indent=2)[:1500])

    available_models = [item.get("id") for item in data.get("data", []) if item.get("id")]
    if not available_models:
        raise RuntimeError("LM Studio is running, but no loaded model was returned. Load a chat/instruct model first.")

    os.environ["MODEL"] = available_models[0]
    print("\nUsing MODEL:", os.environ["MODEL"])

except Exception as error:
    print("Could not reach LM Studio or detect a loaded model.")
    print("Checklist:")
    print("1. Open LM Studio")
    print("2. Load a chat/instruct model")
    print("3. Start the local server")
    print("4. Confirm the server URL is http://localhost:1234/v1")
    print("Error:", error)


In [ ]:
# === Cell 3: Imports and helper functions ===

from __future__ import annotations

from typing import Dict, Any, List, Tuple
from pathlib import Path
import json
import re
import pandas as pd
import requests


def lmstudio_chat(system_prompt: str, user_prompt: str, temperature: float = 0.2, max_tokens: int = 1200) -> str:
    """Direct OpenAI-compatible HTTP call to LM Studio with clearer error messages."""
    url = f"{os.environ['OPENAI_BASE_URL'].rstrip('/')}/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "Content-Type": "application/json",
    }

    model_name = os.environ.get("MODEL", "").strip()
    if not model_name:
        raise RuntimeError("No MODEL is set. Re-run Cell 2 to auto-detect the loaded LM Studio model.")

    payload = {
        "model": model_name,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
        # Some LM Studio models, especially Mistral instruct variants, do not accept
        # the OpenAI-style system role in their chat template. To keep the lab
        # compatible, we fold the system instructions into a single user message.
        "messages": [
            {
                "role": "user",
                "content": (
                    "SYSTEM INSTRUCTIONS:\n"
                    f"{system_prompt.strip()}\n\n"
                    "USER INPUT:\n"
                    f"{user_prompt.strip()}"
                ),
            }
        ],
    }

    resp = requests.post(url, headers=headers, json=payload, timeout=180)

    if not resp.ok:
        # LM Studio usually explains the problem in the response body. Show it.
        raise RuntimeError(
            "LM Studio chat completion failed.\n"
            f"Status: {resp.status_code}\n"
            f"URL: {url}\n"
            f"Model sent: {model_name}\n"
            f"Response body:\n{resp.text[:2000]}"
        )

    data = resp.json()
    return data["choices"][0]["message"]["content"]

def extract_json_object(text: str) -> Dict[str, Any]:
    """Best-effort JSON parser for model outputs."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass
    return {"raw_text": text}


def read_uploaded_text(file_obj) -> str:
    """Read resume/job posting text from txt, pdf, docx, or pasted text file uploads."""
    if file_obj is None:
        return ""

    path = Path(file_obj.name if hasattr(file_obj, "name") else file_obj)
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    if suffix == ".docx":
        import docx
        document = docx.Document(str(path))
        return "\n".join(p.text for p in document.paragraphs)

    return path.read_text(encoding="utf-8", errors="ignore")


def clamp_score(value: Any) -> int:
    try:
        score = int(round(float(value)))
    except Exception:
        return 0
    return max(0, min(100, score))


In [ ]:
# === Cell 4: Four-agent pipeline prompts and functions ===

RESUME_READER_SYSTEM = """
You are Agent 1: Resume Reader.
Extract structured facts from the resume. Do not judge yet. Do not invent missing details.
Return ONLY valid JSON with this schema:
{
  "candidate_name": "string or unknown",
  "target_titles_seen": ["..."],
  "skills": ["..."],
  "tools_technologies": ["..."],
  "experience_summary": ["..."],
  "education": ["..."],
  "certifications": ["..."],
  "projects": ["..."],
  "evidence_bullets": ["short direct evidence from resume"]
}
"""

JOB_ANALYZER_SYSTEM = """
You are Agent 2: Job Posting Analyzer.
Extract the job requirements and hiring signals from the posting.
Return ONLY valid JSON with this schema:
{
  "job_title": "string or unknown",
  "company": "string or unknown",
  "required_skills": ["..."],
  "preferred_skills": ["..."],
  "responsibilities": ["..."],
  "tools_technologies": ["..."],
  "education_or_certifications": ["..."],
  "keywords": ["important ATS keywords"]
}
"""

MATCH_EVALUATOR_SYSTEM = """
You are Agent 3: Resume-to-Job Match Evaluator.
Compare the resume facts to the job facts. Be fair, evidence-based, and conservative.
Do not reward skills that are not supported by resume evidence.
Return ONLY valid JSON with this schema:
{
  "match_score": 0,
  "decision": "Strong match / Possible match / Weak match",
  "matched_requirements": ["..."],
  "partial_matches": ["..."],
  "gaps": ["..."],
  "risk_flags": ["..."],
  "summary": "short explanation"
}
Scoring guidance:
- 90-100: excellent alignment with most required and preferred skills supported by evidence.
- 75-89: good alignment; tailoring is worthwhile; some gaps remain.
- 60-74: possible but below tailoring threshold; notable gaps.
- below 60: weak match.
"""

RESUME_TAILOR_SYSTEM = """
You are Agent 4: Ethical Resume Tailor.
Tailor the resume only using truthful information already present in the resume facts.
Do not invent employers, degrees, tools, dates, metrics, certifications, or experience.
If a useful keyword is not supported by evidence, place it in a "Do not add unless true" section.
Return a polished, job-aligned resume draft in Markdown.
Include these sections:
1. Tailored Professional Summary
2. Core Skills Aligned to the Job
3. Tailored Experience Bullets
4. Projects or Education to Emphasize
5. Keywords to Add Only if True
"""


def agent_1_read_resume(resume_text: str) -> Dict[str, Any]:
    user = f"Resume text:\n\n{resume_text[:12000]}"
    return extract_json_object(lmstudio_chat(RESUME_READER_SYSTEM, user))


def agent_2_analyze_job(job_text: str) -> Dict[str, Any]:
    user = f"Job posting text:\n\n{job_text[:12000]}"
    return extract_json_object(lmstudio_chat(JOB_ANALYZER_SYSTEM, user))


def agent_3_evaluate_match(resume_facts: Dict[str, Any], job_facts: Dict[str, Any]) -> Dict[str, Any]:
    user = f"""
Resume facts JSON:
{json.dumps(resume_facts, indent=2)}

Job facts JSON:
{json.dumps(job_facts, indent=2)}
"""
    result = extract_json_object(lmstudio_chat(MATCH_EVALUATOR_SYSTEM, user))
    result["match_score"] = clamp_score(result.get("match_score", 0))
    return result


def agent_4_tailor_resume(resume_text: str, resume_facts: Dict[str, Any], job_facts: Dict[str, Any], evaluation: Dict[str, Any]) -> str:
    user = f"""
Original resume text:
{resume_text[:12000]}

Resume facts JSON:
{json.dumps(resume_facts, indent=2)}

Job facts JSON:
{json.dumps(job_facts, indent=2)}

Match evaluation JSON:
{json.dumps(evaluation, indent=2)}
"""
    return lmstudio_chat(RESUME_TAILOR_SYSTEM, user, temperature=0.25, max_tokens=2400)


def run_resume_matcher(resume_text: str, job_text: str, threshold: int = 75) -> Dict[str, Any]:
    if not resume_text.strip():
        raise ValueError("Resume text is empty. Upload a resume file or paste resume text.")
    if not job_text.strip():
        raise ValueError("Job posting text is empty. Upload a job posting file or paste job text.")

    resume_facts = agent_1_read_resume(resume_text)
    job_facts = agent_2_analyze_job(job_text)
    evaluation = agent_3_evaluate_match(resume_facts, job_facts)
    score = clamp_score(evaluation.get("match_score", 0))
    should_tailor = score >= int(threshold)

    if should_tailor:
        tailored_resume = agent_4_tailor_resume(resume_text, resume_facts, job_facts, evaluation)
    else:
        tailored_resume = (
            f"### Tailoring skipped\n\n"
            f"The match score was **{score}%**, which is below the selected threshold of **{threshold}%**.\n\n"
            "Before tailoring, improve the resume or choose a more closely aligned job posting. "
            "Review the gaps listed in the evaluation."
        )

    return {
        "resume_facts": resume_facts,
        "job_facts": job_facts,
        "evaluation": evaluation,
        "threshold": int(threshold),
        "should_tailor": should_tailor,
        "tailored_resume": tailored_resume,
    }


In [ ]:
# === Cell 5: Gradio app ===

import gradio as gr

print("Gradio version:", gr.__version__)


def list_to_text(value):
    if isinstance(value, list):
        return "\n".join(f"• {x}" for x in value)
    if value is None:
        return ""
    return str(value)


def evaluation_df(evaluation: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for field in ["matched_requirements", "partial_matches", "gaps", "risk_flags"]:
        values = evaluation.get(field, [])
        if not isinstance(values, list):
            values = [values]
        for item in values:
            rows.append({"category": field.replace("_", " ").title(), "detail": item})
    if not rows:
        rows.append({"category": "Summary", "detail": evaluation.get("summary", "No structured details returned.")})
    return pd.DataFrame(rows)


def run_ui(resume_file, resume_paste, job_file, job_paste, threshold):
    resume_text = (resume_paste or "").strip()
    job_text = (job_paste or "").strip()

    if resume_file is not None:
        resume_text = read_uploaded_text(resume_file).strip() or resume_text
    if job_file is not None:
        job_text = read_uploaded_text(job_file).strip() or job_text

    try:
        result = run_resume_matcher(resume_text, job_text, int(threshold))
    except Exception as error:
        raise gr.Error(f"Resume matcher failed: {error}") from error

    eval_json = result["evaluation"]
    score = clamp_score(eval_json.get("match_score", 0))
    decision = eval_json.get("decision", "Unknown")
    status = "✅ Tailoring completed" if result["should_tailor"] else "⚠️ Tailoring skipped"

    summary_md = f"""
## Match Result: {score}%

**Decision:** {decision}  
**Threshold:** {result['threshold']}%  
**Agent 4 Status:** {status}

**Summary:** {eval_json.get('summary', 'No summary returned.')}
"""

    return (
        json.dumps(result["resume_facts"], indent=2),
        json.dumps(result["job_facts"], indent=2),
        summary_md,
        evaluation_df(result["evaluation"]),
        result["tailored_resume"],
    )


try:
    gr.close_all()
except Exception:
    pass

with gr.Blocks(title="Multi-Agent Resume Matcher", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Multi-Agent Resume Matcher and Tailor\n"
        "Upload or paste a resume and a job posting. Four agents will extract, analyze, score, and—only if the match is strong enough—tailor the resume ethically."
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("## Inputs")
            resume_file = gr.File(label="Upload Resume (.txt, .pdf, .docx)", file_types=[".txt", ".pdf", ".docx"])
            resume_paste = gr.Textbox(label="Or paste resume text", lines=10, placeholder="Paste resume text here...")
            job_file = gr.File(label="Upload Job Posting (.txt, .pdf, .docx)", file_types=[".txt", ".pdf", ".docx"])
            job_paste = gr.Textbox(label="Or paste job posting text", lines=10, placeholder="Paste job posting here...")
            threshold = gr.Slider(minimum=50, maximum=95, value=75, step=5, label="Tailoring Threshold (%)")
            run_btn = gr.Button("Run 4-Agent Resume Match", variant="primary")

        with gr.Column():
            gr.Markdown("## Agent Outputs")
            match_summary = gr.Markdown(label="Match Summary")
            eval_table = gr.Dataframe(label="Agent 3 Evaluation Details", wrap=True, interactive=False)

    with gr.Accordion("Agent 1: Resume Reader JSON", open=False):
        resume_json = gr.Code(language="json", label="Resume Facts")
    with gr.Accordion("Agent 2: Job Analyzer JSON", open=False):
        job_json = gr.Code(language="json", label="Job Facts")
    with gr.Accordion("Agent 4: Tailored Resume Output", open=True):
        tailored_resume = gr.Markdown(label="Tailored Resume")

    run_btn.click(
        fn=run_ui,
        inputs=[resume_file, resume_paste, job_file, job_paste, threshold],
        outputs=[resume_json, job_json, match_summary, eval_table, tailored_resume],
        api_name=False,
    )


demo.launch(
    share=True,
    inline=False,
    inbrowser=False,
    prevent_thread_lock=True,
    show_error=True,
    show_api=False,
)


In [ ]:
# === Cell 6: Optional local test without opening the UI ===
# This uses tiny sample text so students can test the logic before uploading files.

sample_resume = """
Jordan Lee
Data Analyst
Skills: Python, SQL, Excel, Tableau, pandas, data cleaning, dashboard development.
Experience: Built sales dashboards in Tableau, automated weekly reporting with Python and SQL, cleaned customer datasets, and presented findings to managers.
Education: B.S. in Business Analytics.
Project: Customer churn analysis using pandas and logistic regression.
"""

sample_job = """
Data Analyst
We are looking for a Data Analyst with Python, SQL, Tableau, data visualization, dashboarding, and experience communicating insights to business stakeholders.
Preferred: pandas, statistical analysis, customer analytics, and automation experience.
Responsibilities include cleaning data, building dashboards, writing SQL queries, and presenting trends to leadership.
"""

# Uncomment to test after LM Studio is running.
# result = run_resume_matcher(sample_resume, sample_job, threshold=75)
# print(json.dumps(result["evaluation"], indent=2))
# print(result["tailored_resume"])

print("Sample resume/job text is ready. Uncomment the last lines to run a local test.")
